<a href="https://colab.research.google.com/github/Pushpendra-kiet/PES-HMM-DL/blob/main/PES_HMM_DL_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PES-HMM-DL: Prioritized Experience Statistical Hidden Markov Deep Learning Traffic Signal Control

Environment & Dependency Setup

In [ ]:

import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

SUMO_HOME = os.environ.get("SUMO_HOME")
if SUMO_HOME:
    print("SUMO_HOME:", SUMO_HOME)
else:
    print("SUMO_HOME is not set.")
    print("SUMO execution cells require a valid SUMO installation and SUMO_HOME.")


Install Dependencies & Download Data

In [ ]:

import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

SUMO_HOME = os.environ.get("SUMO_HOME")
if SUMO_HOME:
    print("SUMO_HOME:", SUMO_HOME)
else:
    print("SUMO_HOME is not set.")
    print("SUMO execution cells require a valid SUMO installation and SUMO_HOME.")


Download Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("smmmmmmmmmmmm/smart-traffic-management-dataset")

print("Path to dataset files:", path)

Data Preprocessing & HMM Module

In [ ]:

DATA_PATH = Path("/content/smart_traffic_management_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Place the actual Smart Traffic Management "
        "dataset CSV next to this notebook. No synthetic/mock data are generated."
    )

data = pd.read_csv(DATA_PATH)

required_columns = [
    "timestamp",
    "location_id",
    "traffic_volume",
    "avg_vehicle_speed",
    "vehicle_count_cars",
    "vehicle_count_trucks",
    "vehicle_count_bikes",
    "weather_condition",
    "temperature",
    "humidity",
    "accident_reported",
    "signal_status",
]

missing = [c for c in required_columns if c not in data.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
if data["timestamp"].isna().any():
    raise ValueError("Some timestamp values could not be parsed.")

data = data.sort_values("timestamp").reset_index(drop=True)

print("Records:", len(data))
print("Columns:", list(data.columns))
display(data.head())


In [ ]:

HMM_FEATURES = [
    "traffic_volume",
    "avg_vehicle_speed",
    "vehicle_count_cars",
    "vehicle_count_trucks",
]

n = len(data)
train_end = int(0.70 * n)
val_end = train_end + int(0.15 * n)

train_df = data.iloc[:train_end].copy()
val_df = data.iloc[train_end:val_end].copy()
test_df = data.iloc[val_end:].copy()

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[HMM_FEATURES].to_numpy(dtype=np.float64))
X_val = scaler.transform(val_df[HMM_FEATURES].to_numpy(dtype=np.float64))
X_test = scaler.transform(test_df[HMM_FEATURES].to_numpy(dtype=np.float64))

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")


In [ ]:

class TrafficHMM:
    """
    Three-state diagonal-Gaussian HMM.

    States are ordered after fitting by the first feature
    (traffic_volume): low -> moderate -> high.
    """

    def __init__(self, n_states=3, n_features=4, random_state=42):
        self.K = n_states
        self.M = n_features
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)

        self.A = np.full((self.K, self.K), 1.0 / self.K)
        self.pi = np.full(self.K, 1.0 / self.K)
        self.means = None
        self.vars = None
        self.fitted_ = False

    @staticmethod
    def _logsumexp(a, axis=None):
        a = np.asarray(a)
        m = np.max(a, axis=axis, keepdims=True)
        out = m + np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True))
        if axis is not None:
            out = np.squeeze(out, axis=axis)
        return out

    def _log_emission(self, X):
        X = np.asarray(X, dtype=np.float64)
        var = np.maximum(self.vars, 1e-6)

        diff = X[:, None, :] - self.means[None, :, :]
        log_prob = -0.5 * (
            np.sum(np.log(2.0 * np.pi * var)[None, :, :], axis=2)
            + np.sum((diff ** 2) / var[None, :, :], axis=2)
        )
        return log_prob

    def _initialise(self, X):
        # Quantile-based initialisation on traffic_volume (feature 0),
        # followed by small diagonal covariance regularisation.
        q1, q2 = np.quantile(X[:, 0], [1/3, 2/3])
        groups = [
            X[:, 0] <= q1,
            (X[:, 0] > q1) & (X[:, 0] <= q2),
            X[:, 0] > q2,
        ]

        global_mean = X.mean(axis=0)
        global_var = np.var(X, axis=0) + 1e-3

        self.means = np.vstack([
            X[g].mean(axis=0) if np.any(g) else global_mean
            for g in groups
        ])
        self.vars = np.tile(global_var, (self.K, 1))

        # Persistent-state initialisation, matching the intended traffic-state
        # assumption without introducing arbitrary learned transition values.
        self.A = np.full((self.K, self.K), 0.1 / (self.K - 1))
        np.fill_diagonal(self.A, 0.9)
        self.pi = np.full(self.K, 1.0 / self.K)

    def _forward_backward(self, log_emission):
        T = len(log_emission)
        log_A = np.log(np.maximum(self.A, 1e-300))
        log_pi = np.log(np.maximum(self.pi, 1e-300))

        log_alpha = np.empty((T, self.K))
        log_alpha[0] = log_pi + log_emission[0]

        for t in range(1, T):
            log_alpha[t] = (
                log_emission[t]
                + self._logsumexp(log_alpha[t - 1][:, None] + log_A, axis=0)
            )

        log_beta = np.zeros((T, self.K))
        for t in range(T - 2, -1, -1):
            log_beta[t] = self._logsumexp(
                log_A + log_emission[t + 1][None, :] + log_beta[t + 1][None, :],
                axis=1,
            )

        log_likelihood = float(self._logsumexp(log_alpha[-1], axis=0))
        log_gamma = log_alpha + log_beta
        log_gamma -= self._logsumexp(log_gamma, axis=1)[:, None]
        gamma = np.exp(log_gamma)

        xi = np.zeros((T - 1, self.K, self.K))
        for t in range(T - 1):
            log_xi = (
                log_alpha[t][:, None]
                + log_A
                + log_emission[t + 1][None, :]
                + log_beta[t + 1][None, :]
            )
            log_xi -= self._logsumexp(log_xi, axis=None)
            xi[t] = np.exp(log_xi)

        return gamma, xi, log_likelihood

    def fit_em(self, X, max_iter=100, tol=1e-4):
        X = np.asarray(X, dtype=np.float64)
        if X.ndim != 2 or X.shape[1] != self.M:
            raise ValueError(f"Expected X with shape (n_samples, {self.M}).")

        self._initialise(X)
        previous_ll = -np.inf

        for iteration in range(max_iter):
            log_emission = self._log_emission(X)
            gamma, xi, log_likelihood = self._forward_backward(log_emission)

            self.pi = np.maximum(gamma[0], 1e-12)
            self.pi /= self.pi.sum()

            transition_den = gamma[:-1].sum(axis=0) + 1e-12
            self.A = xi.sum(axis=0) / transition_den[:, None]
            self.A = np.maximum(self.A, 1e-12)
            self.A /= self.A.sum(axis=1, keepdims=True)

            state_mass = gamma.sum(axis=0) + 1e-12
            self.means = (gamma.T @ X) / state_mass[:, None]

            diff = X[:, None, :] - self.means[None, :, :]
            self.vars = (
                np.sum(gamma[:, :, None] * diff**2, axis=0)
                / state_mass[:, None]
            )
            self.vars = np.maximum(self.vars, 1e-6)

            if np.isfinite(previous_ll):
                if abs(log_likelihood - previous_ll) < tol:
                    break
            previous_ll = log_likelihood

        # HMM labels are permutation-invariant. Order them by traffic_volume
        # mean so that state 0/1/2 correspond to low/moderate/high traffic.
        order = np.argsort(self.means[:, 0])
        self.means = self.means[order]
        self.vars = self.vars[order]
        self.A = self.A[np.ix_(order, order)]
        self.pi = self.pi[order]
        self.A /= self.A.sum(axis=1, keepdims=True)

        self.fitted_ = True
        self.n_iter_ = iteration + 1
        self.train_log_likelihood_ = log_likelihood
        return self

    def score(self, X):
        if not self.fitted_:
            raise RuntimeError("Fit the HMM before calling score().")
        log_emission = self._log_emission(np.asarray(X, dtype=np.float64))
        _, _, ll = self._forward_backward(log_emission)
        return ll

    def filter_update(self, prev_belief, observation):
        if not self.fitted_:
            raise RuntimeError("Fit the HMM before filtering.")
        observation = np.asarray(observation, dtype=np.float64)
        log_emission = self._log_emission(observation[None, :])[0]

        log_prior = np.log(
            np.maximum(np.asarray(prev_belief, dtype=np.float64) @ self.A, 1e-300)
        )
        log_belief = log_prior + log_emission
        log_belief -= self._logsumexp(log_belief, axis=0)
        return np.exp(log_belief)


Neural Network & Prioritized Replay Buffer

In [ ]:

hmm_model = TrafficHMM(n_states=3, n_features=len(HMM_FEATURES), random_state=SEED)
hmm_model.fit_em(X_train)

print("HMM iterations:", hmm_model.n_iter_)
print("Training log-likelihood:", hmm_model.train_log_likelihood_)
print("\nInitial state probabilities:")
print(hmm_model.pi)

print("\nTransition matrix:")
display(pd.DataFrame(
    hmm_model.A,
    index=["FF", "MO", "CG"],
    columns=["FF", "MO", "CG"],
))

print("\nState means (standardised feature space):")
display(pd.DataFrame(
    hmm_model.means,
    index=["FF", "MO", "CG"],
    columns=HMM_FEATURES,
))


In [ ]:

def filter_sequence(hmm, X):
    belief = np.full(hmm.K, 1.0 / hmm.K)
    beliefs = np.empty((len(X), hmm.K))

    for t, observation in enumerate(X):
        belief = hmm.filter_update(belief, observation)
        beliefs[t] = belief

    return beliefs

train_beliefs = filter_sequence(hmm_model, X_train)
val_beliefs = filter_sequence(hmm_model, X_val)
test_beliefs = filter_sequence(hmm_model, X_test)

belief_columns = ["FF_belief", "MO_belief", "CG_belief"]

print("First five filtered beliefs:")
display(pd.DataFrame(test_beliefs[:5], columns=belief_columns))


6. DQN and HMM-informed prioritized replay

In [ ]:

from dataclasses import dataclass

class DQNNetwork(nn.Module):
    """128-64-32 DQN used by PES-HMM-DL."""
    def __init__(self, input_dim=12, output_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim),
        )

    def forward(self, x):
        return self.net(x)


@dataclass
class Transition:
    state: np.ndarray
    action: int
    reward: float
    next_state: np.ndarray
    done: bool
    z_t: int
    z_next: int


class HMMInformedPER:
    """
    Prioritised replay with:
        p_i = |delta_i| + xi + kappa * A[z_t, z_next]

    Sampling uses p_i**alpha and beta is supplied by the training loop.
    """
    def __init__(self, capacity=50_000, alpha=0.6, kappa=0.1, xi=1e-6):
        self.capacity = capacity
        self.alpha = alpha
        self.kappa = kappa
        self.xi = xi
        self.buffer = []
        self.priorities = np.empty(capacity, dtype=np.float64)
        self.pos = 0

    def __len__(self):
        return len(self.buffer)

    def _priority(self, td_error, z_t, z_next, transition_matrix):
        return (
            abs(float(td_error))
            + self.xi
            + self.kappa * float(transition_matrix[z_t, z_next])
        )

    def push(self, transition, td_error, transition_matrix):
        priority = self._priority(
            td_error, transition.z_t, transition.z_next, transition_matrix
        )

        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.pos] = transition

        self.priorities[self.pos] = priority
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size=64, beta=0.4):
        if len(self.buffer) < batch_size:
            raise ValueError("Not enough transitions in replay buffer.")

        priorities = self.priorities[:len(self.buffer)].copy()
        scaled = np.power(np.maximum(priorities, self.xi), self.alpha)
        probabilities = scaled / scaled.sum()

        indices = np.random.choice(
            len(self.buffer), size=batch_size, replace=False, p=probabilities
        )
        samples = [self.buffer[i] for i in indices]

        weights = np.power(len(self.buffer) * probabilities[indices], -beta)
        weights /= weights.max()

        return samples, indices, torch.as_tensor(weights, dtype=torch.float32)

    def update_priorities(self, indices, td_errors, transition_matrix):
        for idx, td_error in zip(indices, td_errors):
            transition = self.buffer[idx]
            self.priorities[idx] = self._priority(
                td_error,
                transition.z_t,
                transition.z_next,
                transition_matrix,
            )


In [ ]:

class PESHMMDLAgent:
    def __init__(
        self,
        input_dim=12,
        action_dim=4,
        lr=1e-3,
        gamma=0.99,
        replay_capacity=50_000,
        alpha=0.6,
        kappa=0.1,
        xi=1e-6,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.995,
        beta_start=0.4,
        beta_end=1.0,
    ):
        self.gamma = gamma
        self.action_dim = action_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.q_net = DQNNetwork(input_dim, action_dim).to(self.device)
        self.target_net = DQNNetwork(input_dim, action_dim).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.memory = HMMInformedPER(
            capacity=replay_capacity,
            alpha=alpha,
            kappa=kappa,
            xi=xi,
        )

        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.beta_start = beta_start
        self.beta_end = beta_end

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_dim)

        state_tensor = torch.as_tensor(
            state, dtype=torch.float32, device=self.device
        ).unsqueeze(0)

        with torch.no_grad():
            return int(self.q_net(state_tensor).argmax(dim=1).item())

    def beta_at(self, step, total_steps):
        if total_steps <= 1:
            return self.beta_end
        fraction = min(max(step / (total_steps - 1), 0.0), 1.0)
        return self.beta_start + fraction * (self.beta_end - self.beta_start)

    def train_step(self, batch_size, transition_matrix, beta):
        if len(self.memory) < batch_size:
            return None

        samples, indices, weights = self.memory.sample(batch_size, beta)
        weights = weights.to(self.device)

        states = torch.as_tensor(
            np.stack([s.state for s in samples]), dtype=torch.float32, device=self.device
        )
        actions = torch.as_tensor(
            [s.action for s in samples], dtype=torch.long, device=self.device
        ).unsqueeze(1)
        rewards = torch.as_tensor(
            [s.reward for s in samples], dtype=torch.float32, device=self.device
        ).unsqueeze(1)
        next_states = torch.as_tensor(
            np.stack([s.next_state for s in samples]),
            dtype=torch.float32,
            device=self.device,
        )
        dones = torch.as_tensor(
            [s.done for s in samples], dtype=torch.float32, device=self.device
        ).unsqueeze(1)

        q_values = self.q_net(states).gather(1, actions)

        with torch.no_grad():
            next_q = self.target_net(next_states).max(dim=1, keepdim=True).values
            targets = rewards + (1.0 - dones) * self.gamma * next_q

        td_errors = (targets - q_values).detach().squeeze(1).cpu().numpy()

        loss = (
            weights.unsqueeze(1) * (targets - q_values).pow(2)
        ).mean()

        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        self.optimizer.step()

        self.memory.update_priorities(indices, td_errors, transition_matrix)

        return {
            "loss": float(loss.item()),
            "mean_abs_td_error": float(np.mean(np.abs(td_errors))),
        }

    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

    def end_episode(self):
        self.epsilon = max(
            self.epsilon_min,
            self.epsilon * self.epsilon_decay,
        )


SUMO training interface

In [ ]:

def train_on_sumo(
    env,
    hmm,
    episodes=500,
    max_steps=3600,
    batch_size=64,
    target_update_steps=1000,
    seed=42,
):
    """
    Train PES-HMM-DL against an actual SUMO environment.

    The environment must implement:
        reset() -> raw 9-dimensional state
        step(action) -> (raw_next_state, reward, done, info)

    The raw first four features must correspond to the HMM features used
    during fitting. The remaining five state features are retained as the
    DQN state features. The returned DQN input is [9 raw features + 3 beliefs].

    No synthetic states, rewards, or SUMO outputs are generated here.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    agent = PESHMMDLAgent()
    history = []
    global_step = 0

    for episode in range(episodes):
        raw_state = np.asarray(env.reset(), dtype=np.float64)
        if raw_state.shape != (9,):
            raise ValueError(
                f"env.reset() must return a 9-dimensional state; got {raw_state.shape}."
            )

        belief = np.full(hmm.K, 1.0 / hmm.K)
        episode_reward = 0.0

        for _ in range(max_steps):
            state = np.concatenate([raw_state, belief])

            action = agent.select_action(state)
            raw_next_state, reward, done, info = env.step(action)
            raw_next_state = np.asarray(raw_next_state, dtype=np.float64)

            if raw_next_state.shape != (9,):
                raise ValueError(
                    "env.step(action) must return a 9-dimensional next state."
                )

            next_belief = hmm.filter_update(
                belief, raw_next_state[:len(HMM_FEATURES)]
            )
            next_state = np.concatenate([raw_next_state, next_belief])

            z_t = int(np.argmax(belief))
            z_next = int(np.argmax(next_belief))

            transition = Transition(
                state=state.astype(np.float32),
                action=int(action),
                reward=float(reward),
                next_state=next_state.astype(np.float32),
                done=bool(done),
                z_t=z_t,
                z_next=z_next,
            )

            # Initial priority uses the observed one-step reward magnitude.
            # It is replaced by TD-error priority after a training update.
            agent.memory.push(
                transition,
                td_error=abs(float(reward)),
                transition_matrix=hmm.A,
            )

            beta = agent.beta_at(global_step, episodes * max_steps)
            result = agent.train_step(
                batch_size=batch_size,
                transition_matrix=hmm.A,
                beta=beta,
            )

            global_step += 1
            episode_reward += float(reward)
            raw_state = raw_next_state
            belief = next_belief

            if global_step % target_update_steps == 0:
                agent.update_target()

            if done:
                break

        agent.end_episode()
        history.append({
            "episode": episode + 1,
            "reward": episode_reward,
            "epsilon": agent.epsilon,
        })

    return agent, pd.DataFrame(history)


SUMO environment configuration

In [ ]:

# Example configuration check only; no experiment is run here.
SUMO_CONFIG = Path("sumo_config.sumocfg")

if SUMO_CONFIG.exists():
    print("SUMO configuration found:", SUMO_CONFIG.resolve())
else:
    print(
        "SUMO configuration not found. Add the actual .sumocfg and associated "
        "network/route/signal files before running train_on_sumo()."
    )


Reproducibility checks

In [ ]:

agent = PESHMMDLAgent()

print("DQN input dimension:", agent.q_net.net[0].in_features)
print("DQN output/action dimension:", agent.q_net.net[-1].out_features)
print("PER alpha:", agent.memory.alpha)
print("HMM priority weight (kappa):", agent.memory.kappa)
print("PER stabilisation constant (xi):", agent.memory.xi)
print("Beta range:", agent.beta_start, "->", agent.beta_end)

assert agent.q_net.net[0].in_features == 12
assert agent.q_net.net[-1].out_features == 4
assert agent.memory.alpha == 0.6
assert agent.memory.kappa == 0.1
assert agent.memory.xi == 1e-6
assert agent.beta_start == 0.4
assert agent.beta_end == 1.0

print("Implementation checks passed.")


In [ ]:
from scipy import stats

def perform_statistical_tests(proposed, baseline, name="Baseline"):
    # Welch's t-test (unequal variances)
    t_stat, p_val_t = stats.ttest_ind(proposed, baseline, equal_var=False)

    # Wilcoxon signed-rank test (non-parametric paired test)
    w_stat, p_val_w = stats.wilcoxon(proposed, baseline)

    print(f"=== Statistical Comparison: PES-HMM-DL vs {name} ===")
    print(f"Welch's t-test       : t = {t_stat:.4f}, p-value = {p_val_t:.4e}")
    print(f"Wilcoxon Signed-Rank : W = {w_stat:.4f}, p-value = {p_val_w:.4e}")
    if p_val_t < 0.05:
        print(f"Result: Performance improvement over {name} is Statistically Significant (p < 0.05).\n")
    else:
        print(f"Result: No statistically significant difference detected.\n")

perform_statistical_tests(pes_hmm_dl_res, standard_per_res, name="Standard PER")
perform_statistical_tests(pes_hmm_dl_res, dqn_baseline_res, name="Standard DQN")